# Generate visible model traffic from Aurora Lab

**Customer question:** Can a data scientist deploy a model in OpenShift AI, select its endpoint, make real authorized requests, and explain the native graphs?

The working preset calls **aurora-qwen-4b**. To try a model you deployed in the native UI, edit the next cell's `BASE_URL`, `MODEL_ID`, and `AUTH_MODE`. Use its OpenAI-compatible base URL ending in `/v1` and exact served model ID. Creating a deployment does not automatically grant the Workbench access or create a compatible route.

Select **Kernel → Change Kernel → Aurora Inference Demo**. Dependencies live in the persistent `/opt/app-root/src/.venvs/aurora-inference` environment. This notebook creates no orders and has no inventory or RAG tools. Prompts are synthetic Aurora Supply examples; generated explanations require factual review.

- **service_account:** only the exact existing private Gateway on port 8080 and `/ai-showroom/{MODEL_ID}/v1` can receive the rotating Workbench token. Current grants authorize Qwen only. An operator must configure the exact named-model grant and route for another model; the notebook changes no permissions.
- **api_key:** use a verified **HTTPS** endpoint. The cell prompts with hidden `getpass` input and binds the key in memory to that exact URL. Enter the key again after changing the URL. Never place it in code, an environment variable, or notebook output.

The preset's same-namespace Gateway hop is **HTTP**, with native token authentication and named-model authorization. Browser ingress uses separate TLS. No TLS verification is disabled. This private path is distinct from a MaaS subscription demo. Never display `connection()` or its credential dictionary.


In [ ]:
# Edit these three nonsecret settings for your deployed model.
BASE_URL = 'http://showroom-inference-maas-gateway-class.ai-showroom.svc.cluster.local:8080/ai-showroom/aurora-qwen-4b/v1'
MODEL_ID = 'aurora-qwen-4b'
AUTH_MODE = 'service_account'  # Or 'api_key' for a user-selected HTTPS endpoint.

from pathlib import Path
import sys

if Path(sys.prefix).name != 'aurora-inference':
    raise RuntimeError('Select the Aurora Inference Demo kernel before running this notebook.')
helper_dir = Path.cwd() if (Path.cwd() / 'showroom_workbench.py').exists() else Path.cwd() / 'notebooks'
if not (helper_dir / 'showroom_workbench.py').exists():
    raise RuntimeError('Open this notebook from the rhoai-showroom checkout.')
if str(helper_dir) not in sys.path:
    sys.path.insert(0, str(helper_dir))

import pandas as pd
import matplotlib.pyplot as plt
from showroom_workbench import validate_config, prompt_api_key, test_call, manual_load

CONFIG = validate_config({'base_url': BASE_URL, 'model_id': MODEL_ID, 'auth_mode': AUTH_MODE})
API_CREDENTIAL = prompt_api_key(CONFIG) if AUTH_MODE == 'api_key' else None
print('Ready: selected model', CONFIG['model_id'], 'with', CONFIG['auth_mode'], 'authentication.')


## Check the selected model and make one real test call

Run the next cell. First it checks `/models` and requires an exact match for `MODEL_ID`; only then does it send one bounded chat request. The preflight displays sanitized readiness, authorization, network/TLS, or API-compatibility guidance. No upstream error body or credential is displayed. A `403` for a new native model requires an operator to check its exact model grant and route; do not widen permissions yourself.

The preflight has a 10-second limit, followed by a test request limited to 20 seconds and 64 output tokens. The loop requires a successful test for the same endpoint, model, and entered credential. Repeat this cell after changing connection settings. Do not paste credentials into an error report.


In [ ]:
test = await test_call(max_tokens=64, config=CONFIG, api_key=API_CREDENTIAL)
display(pd.DataFrame([test['preflight']]))
display(pd.DataFrame([test['metrics']]))
if test['metrics']['status'] == 'completed':
    print(test['answer'])
else:
    print('Test did not complete. Resolve the displayed preflight or HTTP/network error before starting traffic.')


## Run a short configurable workload

Edit the values below, then run the cell. `INTERACTIONS` is the total request count; `INTERVAL_S` spaces request starts globally. Concurrency defaults to one and is limited to two. Hard limits are **30 requests, 180 seconds, and 256 output tokens per request**; the shorter duration or request limit wins. A failed request stops new traffic; there are no automatic retries.

Use Jupyter's **Interrupt kernel** button to stop early. The helper cancels and joins its client tasks and closes connections before returning; it does not leave a background load generator. A server can still finish a request it already accepted, so stopping the cell does not promise instantaneous GPU inactivity.

Use this gentle default first. These are demonstration requests, not a statistically representative benchmark or proof of an llm-d speedup.


In [ ]:
INTERACTIONS = 6
INTERVAL_S = 2.0
MAX_TOKENS = 96
CONCURRENCY = 1
DURATION_S = 45

if test['metrics']['status'] != 'completed':
    raise RuntimeError('A successful test call is required before starting the loop.')

report = await manual_load(
    interactions=INTERACTIONS,
    interval_s=INTERVAL_S,
    max_tokens=MAX_TOKENS,
    concurrency=CONCURRENCY,
    duration_s=DURATION_S,
    config=CONFIG,
    api_key=API_CREDENTIAL,
    test_result=test,
)
print('Stop reason:', report['stop_reason'])
print('Client tasks remaining:', report['client_tasks_remaining'])
print('Private metrics directory:', report['results_directory'])
assert report['client_tasks_remaining'] == 0


## Inspect the measured results

The test call above is separate from this loop's table and plots. Latency is client-observed, non-streaming elapsed time, including network/authentication/model work; it is **not time to first token**. Token counts come from the provider. Missing usage is left unknown rather than replaced with zero. No cost estimate is fabricated.


In [ ]:
rows = pd.DataFrame(report['rows'])
if rows.empty:
    print('No requests started before the cell stopped.')
else:
    display(rows[['request', 'status', 'http_status', 'elapsed_s',
                  'input_tokens', 'output_tokens', 'total_tokens', 'error_code']])
    complete = rows['status'].eq('completed')
    summary = pd.DataFrame([{
        'Requests started': len(rows),
        'Completed': int(complete.sum()),
        'Other outcomes': int((~complete).sum()),
        'Cell elapsed (s)': report['elapsed_s'],
        'Requests with measured usage': int(rows['total_tokens'].notna().sum()),
        'Measured total tokens': rows['total_tokens'].sum(min_count=1),
    }])
    display(summary)

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    rows['status'].value_counts().plot.bar(ax=axes[0], color='#0066cc', rot=0)
    axes[0].set(title='Actual request outcomes', xlabel='Outcome', ylabel='Requests')
    axes[1].plot(rows['request'], rows['elapsed_s'], marker='o', color='#c9190b')
    axes[1].set(title='Client elapsed time', xlabel='Request', ylabel='Seconds (not TTFT)')
    measured = rows.dropna(subset=['input_tokens', 'output_tokens'])
    if measured.empty:
        axes[2].text(0.5, 0.5, 'Provider usage unavailable', ha='center', va='center')
        axes[2].set_axis_off()
    else:
        measured.set_index('request')[['input_tokens', 'output_tokens']].plot.bar(
            stacked=True, ax=axes[2], color=['#0066cc', '#8fbbf0'], rot=0)
        axes[2].set(title='Provider-reported tokens', xlabel='Request', ylabel='Tokens')
    fig.suptitle('Aurora Supply — measured traffic for ' + CONFIG['model_id'])
    fig.tight_layout()
    plt.show()


## Connect the cell to the native dashboards

In **Observe & monitor → Dashboard**, open **LLM Traffic**, **LLM Utilization**, and **LLM Performance**. For the preset, select project **ai-showroom**, model **aurora-qwen-4b**, and a recent time range containing the UTC start/end shown in the saved metrics. Recheck filters after changing tabs and allow collection/refresh delay.

For another deployed model, select its actual project and model. An external HTTPS provider will not populate this cluster's model graphs unless that provider is separately instrumented.

Look for the actual request burst, input/output token activity, running/waiting requests, and GPU activity on the selected model backends. Other requests can share the model, so those graphs are not exclusively this notebook's measurements. A short CPU/network gap or an idle zero does not establish zero inference latency. If the graph is blank, first check model selection, time window, and collection health.

The private request path can exercise llm-d routing. This notebook does not attribute a speedup to routing, prove cross-node KV transfer, or isolate cache behavior. Use the documented matched benchmarks and picker/backend evidence for those separate claims. Compare the private metrics file with the graph's time window, not with a different run or model.

`metrics.json` and `requests.csv` are written outside the Git checkout under the Workbench's private persistent data directory, with owner-only file permissions. They contain timing/status/token counts and loop settings, **not credentials, prompts, answers, or HTTP headers**. Keep those private files out of Git. Clear notebook outputs before publishing a notebook copy.

After the cell completes, no client tasks or benchmark subprocesses remain. To make another comparison, change one setting, record the model's other traffic, and run one new bounded cell; do not describe such an uncontrolled comparison as causal performance evidence.
